# Import librerie e dataset

In [ ]:
import os
!pwd

os.chdir("../RecSys_Course_AT_PoliMi")

!pwd


In [ ]:
import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline
import tqdm
from tqdm import tqdm
import gc
from xgboost import XGBRanker
import scipy.stats as stats
from sklearn.model_selection import KFold
from scipy.sparse import csr_matrix
from numpy import linalg as LA
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.MatrixFactorization.PureSVDRecommender import ScaledPureSVDRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

from Recommenders.XGBoost.XGBoostRerankerRecommender import XGBoostRerankerRecommender

#from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.m3_model import TripleIntegratedHierarchicalHybridRecommender

from Recommenders.XGBoost.feature_populator import feature_populator

#from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
import Recommenders.hybrid.MultVAEpatch
from Recommenders.hybrid.LinearHybridRecommender import NormalizedLinearCoupleHybridRecommender


In [ ]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [ ]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [ ]:
# Split dataset for train&val
URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage = 0.8)

In [ ]:
evaluator = EvaluatorHoldout(URM_test, cutoff_list=[20])

In [ ]:
best_alpha= 0.15724832635414948
best_beta = 0.08802471282205815
best_gamma = 0.12728641243252908

IALS_Parameters = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595}

SLIMElastic_Parameters = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

EASE_R_Parameters = {
    'topK': 1431,
    'l2_norm': 426.57622242296605}


RP3beta_Parameters = {
    'topK': 35,
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'normalize_similarity': True
}

P3alpha_Parameters = {
    'topK': 91,
    'alpha': 0.10032229,
    'normalize_similarity': True
}

ScaledPureSVD_Parameters = {
    'num_factors' : 152,
    'scaling_items' : 0.000714,
    'scaling_users' : 0.575393
}

ItemKNNCF_Parameters = {
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062
}

MultVAE_params = {
    'learning_rate': 2.2956778160991473e-05,
    'l2_reg': 4.747958042741239e-05, 
    'encoding_size': 597,
    'next_layer_size_multiplier': 2.9916958009201124,
    'dropout': 0.4118674496820792, 
    'anneal_cap': 0.5969296800364489
}

MultVAE_IALS_params = {'alpha': 0.2133024147200015}

os.environ['KMP_DUPLICATE_LIB_OK']='True'
os.environ['MKL_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'

# Training hybrid su URM_Train

In [ ]:
models_to_train = [
    (MultVAERecommender_PyTorch_OptimizerMask, MultVAE_params, "MultVAE"),
    (FeatureCombinedImplicitALSRecommender, IALS_Parameters, "IALS"),
    (SLIMElasticNetRecommender, SLIMElastic_Parameters, "SLIMElastic"),
    (EASE_R_Recommender, EASE_R_Parameters, "EASE_R"),
    (RP3betaRecommender, RP3beta_Parameters, "RP3beta"),
    (P3alphaRecommender, P3alpha_Parameters, "P3alpha"),
    (ScaledPureSVDRecommender, ScaledPureSVD_Parameters, "ScaledPureSVD"),
    (ItemKNNCFRecommender, ItemKNNCF_Parameters, "ItemKNNCF")
]

other_algorithms = {}

for model_class, params, name in models_to_train:
    print(f"--- Training {name} ---")
    
    current_recommender = model_class(URM_train)
    current_recommender.fit(**params)
    
    other_algorithms[name] = current_recommender
    
    print(f"Modello salvato in: other_algorithms['{name}']\n")

# Prova: Ibrido "coppia" tra MultVAE e iALS
multvae_ials_hybrid = NormalizedLinearCoupleHybridRecommender(
    URM_train, 
    [other_algorithms['MultVAE'], other_algorithms['IALS']]
)
multvae_ials_hybrid.fit(alpha=0.2133024147200015)

other_algorithms

In [ ]:
linear_comb_rec = TripleIntegratedHierarchicalHybridRecommender(
    URM_train,
    other_algorithms['SLIMElastic'],
    other_algorithms['EASE_R'],
    other_algorithms['RP3beta'],
    other_algorithms['IALS']
    #other_algorithms['MultVAE_IALS']
)

linear_comb_rec.fit(best_alpha, best_beta, best_gamma)

# Creazione training_dataframe (con label)

In [ ]:
import importlib
import Recommenders.XGBoost.feature_populator
importlib.reload(Recommenders.XGBoost.feature_populator)
from Recommenders.XGBoost.feature_populator import feature_populator

training_dataframe = feature_populator(URM_train, linear_comb_rec, other_algorithms, cutoff = 50)

In [ ]:
URM_validation_coo = sps.coo_matrix(URM_validation)

correct_recommendations = pd.DataFrame({"UserID": URM_validation_coo.row,
                                        "ItemID": URM_validation_coo.col})

training_dataframe = pd.merge(training_dataframe, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
training_dataframe["Label"] = training_dataframe["Exist"] == "both"
training_dataframe.drop(columns = ['Exist'], inplace=True)
training_dataframe

# Creazione di XGBRanker (basato su URM_train)

In [ ]:
groups = training_dataframe.groupby("UserID").size().values
groups

In [ ]:
training_dataframe

In [ ]:
import gc # Garbage Collector interface

# 1. ASSICURARSI DELL'ORDINAMENTO (Cruciale per 'group')
# Sostituisci 'UserID' con la colonna che definisce i tuoi gruppi
#training_dataframe = training_dataframe.sort_values(by="UserID").reset_index(drop=True)

# 2. Creazione X e y
y_train = training_dataframe["Label"]
X_train = training_dataframe.drop(columns=["Label"])

# Conversione tipi (OK come int, ma attenzione se sono univoci)
X_train["UserID"] = X_train["UserID"].astype(int)
X_train["ItemID"] = X_train["ItemID"].astype(int)


# Training hybrid su URM_train_validation

In [ ]:
other_algorithms_f = {}

for model_class, params, name in models_to_train:
    print(f"--- Training {name} ---")
    
    current_recommender = model_class(URM_train_validation)
    current_recommender.fit(**params)
    
    other_algorithms_f[name] = current_recommender
    
    print(f"Modello salvato in: other_algorithms['{name}']\n")

# --- MultVAE_IALS ---
other_algorithms_f['MultVAE_IALS'] = NormalizedLinearCoupleHybridRecommender(
    URM_train, 
    [other_algorithms_f['MultVAE'], other_algorithms_f['IALS']]
)
other_algorithms_f['MultVAE_IALS'].fit(alpha=0.2133024147200015)
# ---

other_algorithms_f

In [ ]:
linear_comb_rec_f = TripleIntegratedHierarchicalHybridRecommender(
    URM_train_validation, 
    other_algorithms_f['SLIMElastic'], 
    other_algorithms_f['EASE_R'], 
    other_algorithms_f['RP3beta'],
    other_algorithms_f['IALS']
    #other_algorithms_f['MultVAE_IALS']
    )
linear_comb_rec_f.fit(best_alpha, best_beta, best_gamma)

In [ ]:
n_users, n_items = URM_train_validation.shape

validation_dataframe = pd.DataFrame(index=range(0,n_users), columns = ["ItemID"])
validation_dataframe.index.name='UserID'

# Creazione validation_dataframe

In [ ]:
validation_dataframe = feature_populator(URM_train_validation, linear_comb_rec_f, other_algorithms_f, cutoff = 50)

In [ ]:
validation_dataframe

In [ ]:
validation_dataframe["UserID"] = validation_dataframe["UserID"].astype(int)
validation_dataframe["ItemID"] = validation_dataframe["ItemID"].astype(int)

# Hyper-parameters tuning con optuna

In [ ]:
from xgboost import XGBRanker

In [ ]:
class XGBoostRerankerRecommender:
    def __init__(self, URM_train, XGB_model, df):
        self.URM_train = URM_train
        self.df = df
        self.XGB_model = XGB_model

    def recommend(self, user_ids, cutoff=20, return_scores=True, remove_seen_flag=True, remove_top_pop_flag=True, remove_custom_items_flag=False):
        recommendations = []
        for user_id in user_ids:
            # print(user_id)
            df_slice = self.df[self.df['UserID'] == user_id]
            items = df_slice.ItemID.to_numpy()
            preds = self.XGB_model.predict(df_slice)
            recommendations.append(items[np.argsort(preds)[-cutoff:][::-1]].tolist())
        
        if return_scores:
            rec, scores = other_algorithms_f['SLIMElastic'].recommend(user_ids, cutoff=cutoff, return_scores=return_scores)
            # useless scores
            return np.array(recommendations), scores
        
        return np.array(recommendations)

    def get_URM_train(self):
        return self.URM_train

In [ ]:
"""def objective_xgboost(trial):

    XGB_model = XGBRanker(
        objective = 'rank:map',
        n_estimators = trial.suggest_int('n_estimators', 1000, 5000, log=True),
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.01, log=True),
        reg_alpha = trial.suggest_float('reg_alpha', 1e-5, 1, log=True),
        reg_lambda = trial.suggest_float('reg_lambda', 1e-5, 1, log=True),
        max_depth = trial.suggest_int('max_depth', 3, 8),
        max_leaves = trial.suggest_int('max_leaves', 32, 850),
        grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        verbosity = 2,
        booster = 'gbtree',
        # tree_method = trial.suggest_categorical('tree_method', ['exact', 'approx', 'hist']),
        tree_method = 'hist',
        gamma = trial.suggest_float('gamma', 1e-1, 12, log=True),
        min_child_weight = trial.suggest_float('min_child_weight', 1e-7, 2, log=True),
        subsample = trial.suggest_float('subsample', 0.2, 0.9),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.2, 0.9),
        # enable_categorical = True
    )

    XGB_model.fit(
        X_train,
        y_train,
        group=groups,
        verbose=True
    )


    # ---- FEATURE IMPORTANCE ----
    %matplotlib inline
    from xgboost import plot_importance

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 25))
    plot_importance(XGB_model, importance_type='weight', title='Weight (Frequence)', max_num_features=100, ax=ax)
    plt.show()

    import pandas as pd
    try:
        importance_dict = XGB_model.get_score(importance_type='weight')
    except AttributeError:
        importance_dict = XGB_model.get_booster().get_score(importance_type='weight')
    importance_df = pd.DataFrame({
        'Feature': list(importance_dict.keys()),
        'Importance': list(importance_dict.values())
    }).sort_values(by='Importance', ascending=False)
    with pd.option_context('display.max_rows', None):
        print(importance_df)
    # -------------

    recommender = XGBoostRerankerRecommender(URM_train_validation, XGB_model, validation_dataframe)
    result_df, _ = evaluator.evaluateRecommender(recommender)

    del recommender
    gc.collect()

    return result_df.loc[20, 'RECALL']"""


def objective_xgboost(trial):

    XGB_model = XGBRanker(
        objective = 'rank:map',
        n_estimators = trial.suggest_int('n_estimators', 1000, 5000, log=True),
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.01, log=True),
        reg_alpha = trial.suggest_float('reg_alpha', 1e-5, 1, log=True),
        reg_lambda = trial.suggest_float('reg_lambda', 1e-5, 1, log=True),
        max_depth = trial.suggest_int('max_depth', 3, 8),
        max_leaves = trial.suggest_int('max_leaves', 32, 850),
        grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        verbosity = 2,
        booster = 'gbtree',
        tree_method = 'hist',
        gamma = trial.suggest_float('gamma', 1e-1, 12, log=True),
        min_child_weight = trial.suggest_float('min_child_weight', 1e-7, 2, log=True),
        subsample = trial.suggest_float('subsample', 0.2, 0.9),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.2, 0.9),
    )

    XGB_model.fit(
        X_train,
        y_train,
        group=groups,
        verbose=False
    )

    recommender = XGBoostRerankerRecommender(URM_train_validation, XGB_model, validation_dataframe)
    result_df, _ = evaluator.evaluateRecommender(recommender)
    
    current_recall = result_df.loc[20, 'RECALL']

    # --- LOGICA PER IL PLOT DEL BEST TRIAL ---
    is_best = False
    try:
        # Se il valore corrente supera il migliore registrato finora
        if current_recall > trial.study.best_value:
            is_best = True
    except ValueError:
        # Se è il primo trial in assoluto, best_value solleva ValueError
        is_best = True

    if is_best:
        print(f"\n[NEW BEST] Trial {trial.number} found! Recall@20: {current_recall:.5f}")
        
        %matplotlib inline
        from xgboost import plot_importance
        import matplotlib.pyplot as plt
        import pandas as pd

        # Plot grafico
        fig, ax = plt.subplots(figsize=(10, 15)) # Ridotto leggermente per leggibilità
        plot_importance(XGB_model, importance_type='weight', title=f'Best Trial {trial.number} - Weight Importance', max_num_features=50, ax=ax)
        plt.show()

        # Print tabella importance
        try:
            importance_dict = XGB_model.get_score(importance_type='weight')
        except AttributeError:
            importance_dict = XGB_model.get_booster().get_score(importance_type='weight')
            
        importance_df = pd.DataFrame({
            'Feature': list(importance_dict.keys()),
            'Importance': list(importance_dict.values())
        }).sort_values(by='Importance', ascending=False)
        
        print(importance_df.head(20)) # Stampiamo le top 20
    # ------------------------------------------

    del recommender
    gc.collect()

    return current_recall

In [ ]:
import optuna

In [ ]:
study = optuna.create_study(direction='maximize', study_name='xgboost_tuning_fixed_more_recommenders_more_candidates')
study.optimize(objective_xgboost, n_trials=200)

# Parametri finali xgboost

In [ ]:
XGBoost_Parameters = {    
    'objective': 'rank:map', 
    'n_estimators': 4311, 
    'learning_rate': 0.007651019829692804, 
    'reg_alpha': 0.00046969890359217103, 
    'reg_lambda': 0.00036892979762148984, 
    'max_depth': 5, 
    'max_leaves': 72, 
    'grow_policy': 'depthwise', 
    'gamma': 0.35323759745948785, 
    'min_child_weight': 2.0095902489610272e-05, 
    'subsample': 0.36462843836663134, 
    'colsample_bytree': 0.3338747645086767
}

# Creazione final_train_dataframe

In [ ]:
final_train_dataframe = feature_populator(URM_train_validation, linear_comb_rec_f, other_algorithms_f, cutoff = 50)

In [ ]:
URM_test_coo = sps.coo_matrix(URM_test)

correct_recommendations_all = pd.DataFrame({"UserID": URM_test_coo.row,
                                        "ItemID": URM_test_coo.col})
correct_recommendations_all

In [ ]:
final_train_dataframe = pd.merge(final_train_dataframe, correct_recommendations_all, on=['UserID','ItemID'], how='left', indicator='Exist')
final_train_dataframe["Label"] = final_train_dataframe["Exist"] == "both"
final_train_dataframe.drop(columns = ['Exist'], inplace=True)
final_train_dataframe

# Training finale XGBoost

In [ ]:
groups = final_train_dataframe.groupby("UserID").size().values
groups

In [ ]:
from xgboost import XGBRanker
XGB_model_f = XGBRanker(**XGBoost_Parameters)

y_train = final_train_dataframe["Label"]
X_train = final_train_dataframe.drop(columns=["Label"])
X_train["UserID"] = X_train["UserID"].astype(int)
X_train["ItemID"] = X_train["ItemID"].astype(int)

print(np.isinf(X_train).sum())  # Count of infinite values in training data


# Check for NaN values
print(np.isnan(X_train).sum())  # Count of NaN values in training data

X_train = np.nan_to_num(X_train, posinf=np.finfo(np.float32).max, neginf=0.0)

XGB_model_f.fit(
    X_train,
    y_train,
    group=groups,
    verbose=True
)

In [ ]:
%matplotlib inline
from xgboost import plot_importance

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 25))
plot_importance(XGB_model_f, importance_type='weight', title='Weight (Frequence)', max_num_features=100, ax=ax)
plt.show()

import pandas as pd
try:
    importance_dict = XGB_model_f.get_score(importance_type='weight')
except AttributeError:
    importance_dict = XGB_model_f.get_booster().get_score(importance_type='weight')
importance_df = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importance': list(importance_dict.values())
}).sort_values(by='Importance', ascending=False)
with pd.option_context('display.max_rows', None):
    print(importance_df)

# Training modelli con URM_all

In [ ]:
other_algorithms_all = {}

# Loop per trainare e valutare tutti i modelli
for model_class, params, name in models_to_train:
    print(f"--- Training {name} ---")
    
    current_recommender = model_class(URM_all)
    current_recommender.fit(**params)
    
    other_algorithms_all[name] = current_recommender
    
    print(f"Modello salvato in: other_algorithms_all['{name}']\n")

other_algorithms_all['MultVAE_IALS'] = NormalizedLinearCoupleHybridRecommender(
    URM_train, 
    [other_algorithms_all['MultVAE'], other_algorithms_all['IALS']]
)
other_algorithms_all['MultVAE_IALS'].fit(alpha=0.2133024147200015)

other_algorithms_all

In [ ]:
linear_comb_rec_all = TripleIntegratedHierarchicalHybridRecommender(
    URM_all,
    other_algorithms_all['SLIMElastic'],
    other_algorithms_all['EASE_R'],
    other_algorithms_all['RP3beta'],
    other_algorithms_all['IALS']
    #other_algorithms_all['MultVAE_IALS']
    )
linear_comb_rec_all.fit(best_alpha, best_beta, best_gamma)

# Creazione prediction_dataframe

In [ ]:
prediction_dataframe = feature_populator(URM_all, linear_comb_rec_all, other_algorithms_all, cutoff = 50)
prediction_dataframe

# Run finale XGBoostRerankerRecommender + submission

In [ ]:
prediction_dataframe["UserID"] = prediction_dataframe["UserID"].astype(int)
prediction_dataframe["ItemID"] = prediction_dataframe["ItemID"].astype(int)

In [ ]:
recommender = XGBoostRerankerRecommender(URM_all, XGB_model_f, prediction_dataframe)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = recommender.recommend([user_id], cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations[0][0]))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)

df_recommendations.to_csv("../Results/final csvs/recommendations_xgboost_31122025.csv", index=False)

end_time = time.time()